#**Task 1: Implement Quantization from Scratch**

In [19]:
def quantize(data):
    q_min = -128
    q_max = 127

    # Find minimum and maximum values
    x_min = min(data)
    x_max = max(data)

    # Calculate scale
    scale = (x_max - x_min) / (q_max - q_min)
    z=0

    # Quantize values
    quantized = []

    for x in data:
        q = round(x / scale) + z


        q = max(q_min, min(q_max, q))

        quantized.append(q)

    return quantized, scale, z


# Test array
data = [-1.0, -0.5, 0.0, 0.5, 1.0, 0.72]

quantized, scale, z = quantize(data)

print("Original values :", data)
print("Quantized values:", quantized)
print("Scale           :", scale)
print("Zero-point      :", z)

Original values : [-1.0, -0.5, 0.0, 0.5, 1.0, 0.72]
Quantized values: [-128, -64, 0, 64, 127, 92]
Scale           : 0.00784313725490196
Zero-point      : 0


The FP32 values were converted into INT8 values using a scale factor with zero-point. The quantized values require less memory and reduces the cost of GPU's

#**Task 2: Estimate Model Size from Parameter Count**

In [20]:
def model_size(parameters, dtype):

    bytes_per_parameter = {"fp32": 4, "fp16": 2,"int8": 1}

    size_bytes = parameters * bytes_per_parameter[dtype]
    size_mb = size_bytes / (1024 * 1024)
    size_gb = size_bytes / (1024 * 1024 * 1024)

    print(dtype.upper(), "Model Size:")
    print(round(size_mb, 2), "MB")
    print(round(size_gb, 2), "GB")
    print()


# 7 Billion parameter model
parameters = 7_000_000_000

model_size(parameters, "fp32")
model_size(parameters, "fp16")
model_size(parameters, "int8")

FP32 Model Size:
26702.88 MB
26.08 GB

FP16 Model Size:
13351.44 MB
13.04 GB

INT8 Model Size:
6675.72 MB
6.52 GB



As the precision decreases from FP32 to INT8, the model size decreases significantly. FP32 requires 4 bytes per parameter, while INT8 requires only 1 byte per parameter, resulting in approximately 75% reduction in weight storage.

#**Task 3: System Design Scenario**


##1. Where would you split the model (within a node vs across nodes), and why?

If the model is too big to fit on one GPU, I would first try to distribute it among the 8 GPUs in the same node. I would prefer this because the GPUs inside the same node can communicate with each other using NVLink, which is very fast.

If the model is still too large even after using all 8 GPUs, then I would move to multiple nodes and split the model across them using InfiniBand for communication.


##2. What role does NVLink play vs InfiniBand in this setup?

NVLink is mainly used when the GPUs are inside the same node. It allows the GPUs to exchange data quickly while working together on the model.

InfiniBand is used when we have multiple nodes. It connects the different servers and allows the GPUs in one node to communicate with GPUs in another node.

In simple words:

NVLink → communication within the same node
InfiniBand → communication between different nodes


##3. What would go wrong if you swapped their roles?

If we used InfiniBand for communication between GPUs that are already in the same node, it would not make much sense because NVLink is available for much faster communication. It could add extra overhead and increase the time needed for the GPUs to exchange data.

Similarly, NVLink cannot simply be used to connect GPUs that are located in different servers. For communication between separate nodes, we need a network connection such as InfiniBand.

This matters because during model inference, GPUs may have to exchange data many times. If this communication becomes slow, the GPUs might have to wait for each other. As a result, the overall inference can become slower.


##4. Write your answer as a short design note (half a page), no code needed.
Design Note

If a model is too large to fit on a single GPU, I would first try to split it across the 8 GPUs available in the same node. I would prefer this setup because the GPUs can communicate through NVLink, which provides fast GPU-to-GPU communication. Keeping the model within one node as much as possible can help reduce communication delays during inference.

If the model is still too large after using all the GPUs in one node, I would then use multiple nodes. In that case, InfiniBand would be used to connect the GPUs across the different nodes.

Basically, I would use NVLink for communication between GPUs in the same node and InfiniBand for communication between different nodes. These connections are designed for different purposes, so using them this way should give better performance.

If we used InfiniBand between GPUs within the same node, we would be ignoring the faster NVLink connection and could introduce unnecessary communication overhead. Also, NVLink is not a replacement for the network connection needed between separate servers.

So, my approach would be to use all the GPUs in one node first and only add more nodes when the model cannot fit, while using NVLink for intra-node communication and InfiniBand for inter-node communication.\

In [22]:
def check(flops, gpu, memory, bandwidth):

    time_compute = flops / gpu
    time_memory = memory/ memory

    if time_compute > time_memory:
        result = "Compute-bound"
    elif time_memory > time_compute:
        result = "Memory-bound"
    else:
        result = "Balanced"

    return result, time_compute, time_memory


# Configuration 1
result1 = check(200e9, 20e12, 10e9, 1e12)

print("Configuration 1")
print("Inference type:", result1[0])
print("Compute time:", round(result1[1] * 1000, 2), "ms")
print("Memory time :", round(result1[2] * 1000, 2), "ms")
print()


# Configuration 2
result2 = check(500e9, 20e12, 10e9, 1e12)

print("Configuration 2")
print("Inference type:", result2[0])
print("Compute time:", round(result2[1] * 1000, 2), "ms")
print("Memory time :", round(result2[2] * 1000, 2), "ms")
print()


# Configuration 3
result3 = check(100e9, 20e12, 20e9, 1e12)

print("Configuration 3")
print("Inference type:", result3[0])
print("Compute time:", round(result3[1] * 1000, 2), "ms")
print("Memory time :", round(result3[2] * 1000, 2), "ms")

Configuration 1
Inference type: Memory-bound
Compute time: 10.0 ms
Memory time : 1000.0 ms

Configuration 2
Inference type: Memory-bound
Compute time: 25.0 ms
Memory time : 1000.0 ms

Configuration 3
Inference type: Memory-bound
Compute time: 5.0 ms
Memory time : 1000.0 ms


The comparison helps us understand whether the main limitation is computation or memory access, so we can choose a suitable optimization such as improving compute efficiency or reducing memory usage.